# League of Legends Rhythm Analysis: Concise Version

This notebook is the teaching front end for the Python workflow. It does not copy all of `riot_analysis.py`; instead it shows how to use the reusable functions to run the same analysis in a shorter, auditable way.

The heavy implementation lives in:

- `riot_analysis.py`: per-server rhythm, PCA, win-rate, DeltaMMR, phase, and circular-model analysis
- `grand_analysis.py`: across-server GRAND summaries, density plots, BIC comparisons, and HTML report generation
- `Parquet_longerAnalyses_May_26.py`: command-line runner for all servers

## 1. Setup

Run this notebook from the repository root or from the `notebooks/` folder. The path setup below handles either case.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import HTML, IFrame, Image, display

ROOT = Path.cwd()
if not (ROOT / "riot_analysis.py").exists() and (ROOT.parent / "riot_analysis.py").exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

print(f"Repository root: {ROOT}")

In [ ]:
from riot_analysis import AnalysisConfig, available_platforms, connect_analysis_database, run_platform_analysis
from grand_analysis import run_grand_analysis

DB_FILE = ROOT / "riot_local.duckdb"
PARQUET_FILE = None  # Auto-detect RIOT_DB_PATH/RIOT_PARQUET_PATH, this server, or Colab Drive.
OUTPUT_ROOT = ROOT / "results"

print("DuckDB cache found:", DB_FILE.exists())
print("Raw Parquet source:", "auto-detect" if PARQUET_FILE is None else PARQUET_FILE)
print("Output folder:", OUTPUT_ROOT)

## 2. Inspect the Included Final Report

The release repository includes the final GRAND report and figures. These can be viewed without the DuckDB data file.

In [ ]:
report_path = OUTPUT_ROOT / "GRAND" / "grand_final_report.html"

if report_path.exists():
    display(HTML(f'<p><a href="{report_path.relative_to(ROOT)}" target="_blank">Open the final GRAND HTML report</a></p>'))
    display(IFrame(src=str(report_path.relative_to(ROOT)), width="100%", height=720))
else:
    print("No final report found. Regenerate it after running the GRAND analysis.")

## 3. Read Key GRAND Tables

These tables summarize the final N-weighted across-server analysis.

In [ ]:
grand_dir = OUTPUT_ROOT / "GRAND"
for filename in [
    "grand_metric_summary.csv",
    "grand_phase_summary.csv",
    "grand_pc_peak_density_modes.csv",
    "grand_pooled_circular_bimodality.csv",
]:
    path = grand_dir / filename
    print(f"\n--- {filename} ---")
    if path.exists():
        display(pd.read_csv(path))
    else:
        print("Missing:", path)

## 4. Run One Server

This is the smallest complete rerun. It will create or refresh the local `riot_local.duckdb` cache from the server/Colab Parquet source, produce per-server figures/tables under `results/<SERVER>/`, and return a one-row summary.

Set `RUN_ANALYSIS = True` only when the raw Riot Parquet data are available on this server or in the shared Colab Drive.

In [ ]:
RUN_ANALYSIS = False
PLATFORM = "EUW1"

if RUN_ANALYSIS:
    conn = connect_analysis_database(DB_FILE, parquet_file=PARQUET_FILE)
    try:
        print("Available platforms:", available_platforms(conn))
        config = AnalysisConfig(
            platform=PLATFORM,
            output_root=OUTPUT_ROOT,
            top_n_players=1000,
            n_jobs=8,
        )
        summary = run_platform_analysis(conn, config)
        display(pd.DataFrame([summary]))
    finally:
        conn.close()
else:
    print("Skipped. Set RUN_ANALYSIS = True to run one server.")

## 5. Run All Servers from the Command Line

For a full rerun, the command-line script is clearer than a notebook loop because it reports progress server by server and writes a combined summary.

In [ ]:
RUN_ALL_SERVERS = False

if RUN_ALL_SERVERS:
    import subprocess
    command = [sys.executable, "Parquet_longerAnalyses_May_26.py", "--output-root", str(OUTPUT_ROOT)]
    subprocess.run(command, cwd=ROOT, check=True)
else:
    print("Skipped. Set RUN_ALL_SERVERS = True to run every server.")

## 6. Regenerate the GRAND Analysis

This reads existing per-server outputs and rebuilds `results/GRAND/`, including the HTML report and pooled circular BIC comparisons.

In [ ]:
RUN_GRAND = False

if RUN_GRAND:
    result = run_grand_analysis(output_root=OUTPUT_ROOT)
    display(result)
else:
    print("Skipped. Set RUN_GRAND = True after per-server outputs exist.")

## 7. Display Final Figures

These are the key final figures from the GRAND analysis.

In [ ]:
for filename in [
    "grand_pooled_circular_bimodality_fits.png",
    "grand_pc_peak_density.png",
    "grand_within_subject_summary.png",
    "grand_performance_pca_loadings.png",
    "grand_success_aware_pca_loadings.png",
    "grand_win_rate_by_local_hour.png",
]:
    path = grand_dir / filename
    print(f"\n--- {filename} ---")
    if path.exists():
        display(Image(filename=str(path)))
    else:
        print("Missing:", path)

## 8. How This Maps to `riot_analysis.py`

The concise workflow above calls the same functions used by the full script:

- `run_platform_analysis(...)` handles the per-server workflow.
- `AnalysisConfig(...)` controls the server, output folder, player count, and parallelism.
- `run_grand_analysis(...)` builds the final across-server report.

Students who want implementation detail should read `riot_analysis.py` function by function rather than scrolling through a monolithic notebook.